# 01 - Analyse Exploratoire des Données

Ce notebook couvre la phase **Compréhension des données** de CRISP-DM. Il est volontairement très annoté afin de montrer clairement les étapes attendues pour l’évaluation académique.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT / 'src'))

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from feature_engineering import load_streaming_dataset

## Chargement et Adaptation du Dataset

Le dataset IBM Telco est chargé depuis `data/raw`, puis adapté au contexte d’une plateforme de streaming vidéo. Les variables télécom sont renommées en variables métier streaming.

In [ ]:
df = load_streaming_dataset(ROOT / 'data' / 'raw' / 'IBM_Telco_Customer_Churn.csv')
df.head()

## Vue d’Ensemble du Dataset

Cette section vérifie le nombre de lignes, le nombre de colonnes, les types de données et les premières observations.

In [ ]:
print(f'Lignes: {df.shape[0]:,}')
print(f'Colonnes: {df.shape[1]:,}')
df.info()

## Analyse des Valeurs Manquantes

La variable `TotalCharges`, renommée `total_spent`, contient des valeurs vides dans le dataset original. Elles sont converties en valeurs manquantes numériques.

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]

## Vérification des Doublons

`customer_id` doit être unique. Des doublons pourraient fausser les indicateurs et l’apprentissage.

In [ ]:
df['customer_id'].duplicated().sum()

## Statistiques Descriptives

Les statistiques descriptives permettent d’analyser les distributions, les valeurs extrêmes et la dispersion des variables numériques.

In [ ]:
df.describe().T

## Distribution de la Variable Cible

Le churn est une classe minoritaire. Cette observation justifie le split stratifié et l’utilisation de SMOTE uniquement sur l’entraînement.

In [ ]:
target_dist = df['churn'].value_counts(normalize=True).rename('pourcentage') * 100
print(target_dist)
sns.countplot(data=df, x='churn')
plt.title('Distribution du churn')
plt.show()

## Heatmap de Corrélation

La heatmap permet d’observer les relations entre les variables numériques, notamment l’ancienneté, le prix, les dépenses et l’engagement.

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns
plt.figure(figsize=(10, 7))
sns.heatmap(df[numeric_cols].corr(), cmap='coolwarm', center=0)
plt.title('Heatmap de corrélation')
plt.show()

## Histogrammes

Les histogrammes montrent la forme des distributions pour l’ancienneté, le prix, les dépenses et les variables de comportement.

In [ ]:
df[['subscription_months', 'monthly_subscription_fee', 'total_spent', 'views_per_week', 'average_watch_time']].hist(figsize=(12, 8), bins=30)
plt.tight_layout()
plt.show()

## Boxplots par Churn

Les boxplots comparent les churners et non-churners sur les variables numériques importantes.

In [ ]:
for col in ['subscription_months', 'monthly_subscription_fee', 'total_spent', 'engagement_score']:
    plt.figure(figsize=(7, 4))
    sns.boxplot(data=df, x='churn', y=col)
    plt.title(f'{col} selon le churn')
    plt.show()

## Analyse du Churn par Catégorie

Ces graphiques relient le churn à des dimensions exploitables par les équipes Marketing et CRM.

In [ ]:
for col in ['subscription_plan', 'streaming_quality', 'payment_method', 'offline_downloads', 'premium_support', 'favorite_genre']:
    rates = df.groupby(col)['churn'].mean().sort_values(ascending=False).reset_index()
    plt.figure(figsize=(9, 4))
    sns.barplot(data=rates, x='churn', y=col, color='#2f6f9f')
    plt.title(f'Taux de churn par {col}')
    plt.xlabel('Taux de churn')
    plt.ylabel(col)
    plt.show()

## Conclusion EDA

Les signaux attendus sont l’ancienneté faible, le plan mensuel, la méthode de paiement, le prix mensuel et l’engagement. Ces résultats orientent la préparation des données et la modélisation.